# Notebook 3 — Non-CMF Unlearning Methods on 3:7 Cross-Class Split

**Experiment:** ReGUn benchmark — Group A (non-CMF unlearning methods).

**Prerequisite:** Run **Notebook 1** first and attach its output dataset.
Set `CKPT_DATASET_DIR` below to the Notebook 1 dataset mount path.

This notebook runs all non-CMF unlearning methods on the fixed 3:7 stratified
cross-class forget split generated in Notebook 1.

**Methods:** Fine-tune, NegGrad+, Random-label, SalUn, SVD, UNSIR, SCRUB

| Stage | Description |
|-------|-------------|
| **A** | Environment setup, load config |
| **B** | Load split files, dataset, original model |
| **C** | Run all non-CMF unlearning methods (≥ 3 seeds × 7 methods) |
| **D** | Evaluate: output/probe/NCC retain+forget accuracy |
| **E** | Results CSV + summary table |

## A. Environment Setup & Load Config

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math, time, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SET THIS to the Kaggle dataset mount path from Notebook 1.
# ══════════════════════════════════════════════════════════════════════
CKPT_DATASET_DIR = '/kaggle/input/regun-notebook1'  # ← EDIT THIS

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/regun_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/regun/regun_config.json',
    f'{CKPT_DATASET_DIR}/regun/regun_config.json',
]

config_path = None
CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p
        CKPT_ROOT_NB1 = os.path.dirname(_p)
        break

assert config_path is not None, (
    'regun_config.json not found. Checked:\n' +
    '\n'.join(f'  - {p}' for p in _CONFIG_CANDIDATES))

with open(config_path) as f:
    CFG = json.load(f)

TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG['TEST_FRACTION']
_MODE_TAG         = CFG['_MODE_TAG']
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
SPLIT_SEEDS       = CFG['SPLIT_SEEDS']
FORGET_FRACTION   = CFG['FORGET_FRACTION']
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

_old_root = CFG['CKPT_ROOT']
def _repath(p): return p.replace(_old_root, CKPT_ROOT_NB1)
CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])
SPLIT_DIR = _repath(CFG['SPLIT_DIR'])

for p, name in [(CKPT_PRETRAIN, 'pre_train')]:
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {name}: {p}')

DATA_PATH  = '/kaggle/working/data'
WORK_ROOT  = '/kaggle/working'
CKPT_ROOT_NB3 = '/kaggle/working/checkpoints/regun_nb3'
os.makedirs(CKPT_ROOT_NB3, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

# ── Method cap ────────────────────────────────────────────────────────
MAX_EPOCHS = 1 if TEST_MODE else 50  # ≤ 50 epoch cap per method

print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  Split seeds: {SPLIT_SEEDS}')

## B. Load Dataset, Split Files & Original Model

In [ ]:
from utils import get_dataset, get_model, test, load_encoder_ckpt_safely, SubSet
from unlearn import unlear_func
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

UNLEARN_BS = 8 if TEST_MODE else 128

# ── Learning rate table (same as existing notebooks) ──────────────────
_LR = {
    'random_label':        {'cifar10':{'s':1e-2},'cifar100':{'s':3e-3},'tinyimagenet':{'s':5e-4}},
    'salun':               {'cifar10':{'s':1e-2},'cifar100':{'s':3e-3},'tinyimagenet':{'s':5e-4}},
    'grad_ascent_descent': {'cifar10':{'s':1e-3},'cifar100':{'s':5e-5},'tinyimagenet':{'s':5e-4}},
    'scrub':               {'cifar10':{'s':1e-4},'cifar100':{'s':1e-3},'tinyimagenet':{'s':5e-3}},
    'tarun':               {'cifar10':{'s':2e-3},'cifar100':{'s':3e-5},'tinyimagenet':{'s':2e-5}},
    'SVD':                 {'cifar10':{'s':1e-2},'cifar100':{'s':1e-2},'tinyimagenet':{'s':1e-3}},
}
_EPOCHS = {
    'random_label': 3, 'salun': 3, 'grad_ascent_descent': 3,
    'scrub': 3, 'tarun': 3, 'SVD': 50,
}
if TEST_MODE:
    _EPOCHS = {k: 1 for k in _EPOCHS}

_SVD = {
    'cifar10':     {'alpha_r': 100, 'alpha_f': 3,  'samples': 900,  'max_patches': 10000},
    'cifar100':    {'alpha_r': 1000,'alpha_f': 30, 'samples': 990,  'max_patches': 10000},
    'tinyimagenet':{'alpha_r': 30,  'alpha_f': 10, 'samples': 999,  'max_patches': 10000},
}
if TEST_MODE:
    _SVD = {k: {'alpha_r':2,'alpha_f':1,'samples':4,'max_patches':10} for k in _SVD}

def get_lr(method):
    return _LR.get(method, {}).get(DATASET, {}).get('s', 1e-3)

def get_epochs(method):
    ep = _EPOCHS.get(method, 1 if TEST_MODE else 3)
    return min(ep, MAX_EPOCHS)  # enforce 50-epoch cap

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=1.0, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='regun',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_targets = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_targets[i] for i in kept]
        return sub

    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: {TEST_FRACTION*100:.1f}% → '
          f'Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=UNLEARN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2,
                 pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(dataset_test, **TEST_KW)

# ── Load split files ─────────────────────────────────────────────────
splits = {}
for seed in SPLIT_SEEDS:
    split_file = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    if not os.path.exists(split_file):
        raise FileNotFoundError(f'Split file missing: {split_file}')
    with open(split_file) as f:
        splits[seed] = json.load(f)
    print(f'Seed {seed}: forget={splits[seed]["n_forget"]}  retain={splits[seed]["n_retain"]}')

# ── Load original Θ_o ─────────────────────────────────────────────────
args_pt = make_args(unlearn_method='pre_train', remove_FC=False, CMFClassifier=True)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
print('\n── Original model accuracy ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')

## B-helper. Updated Eval Harness (Index-Based)

Since the forget set spans ALL classes (cross-class split), the eval harness must
compute retain/forget accuracy over SAMPLE INDEX SETS rather than class labels.

In [ ]:
def eval_on_indices(model, dataset, indices, device, batch_size=256):
    """
    Evaluate model accuracy over an arbitrary set of sample indices.
    Used by the cross-class 3:7 split eval harness.
    Returns (accuracy, n_correct, n_total).
    """
    if len(indices) == 0:
        return 0.0, 0, 0
    subset = SubSet(dataset, indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    acc = correct / max(1, total)
    return acc, correct, total


def run_probe_eval(model, train_indices, forget_indices, retain_indices,
                   dataset_train, dataset_test, device, num_classes, args):
    """
    Run linear probe evaluation over the cross-class split.
    Trains a linear probe on RETAIN train features, evaluates on forget/retain test indices.
    Returns dict with probe_retain_acc, probe_forget_acc.
    """
    try:
        from evaluation.linear_prob import unified_linear_probe
        retain_train_ds = SubSet(dataset_train, retain_indices)
        retain_train_loader = torch.utils.data.DataLoader(
            retain_train_ds, batch_size=256, shuffle=True,
            num_workers=2, pin_memory=True)
        out = unified_linear_probe(
            args=args, model=model,
            train_loader=retain_train_loader,
            test_loader=torch.utils.data.DataLoader(
                dataset_test, batch_size=256, shuffle=False,
                num_workers=2, pin_memory=True),
            device=device, num_classes=num_classes,
        )
        return {
            'probe_retain_acc': out.get('acc_test_retain', float('nan')),
            'probe_forget_acc': out.get('acc_test_forget', float('nan')),
        }
    except Exception as e:
        print(f'  [probe eval failed: {e}]')
        return {'probe_retain_acc': float('nan'), 'probe_forget_acc': float('nan')}


print('Eval harness defined.')

## C. Run Non-CMF Unlearning Methods

In [ ]:
# ── Group A: all non-CMF methods ──────────────────────────────────────
RUN_METHODS = [
    'grad_ascent_descent',  # NegGrad+
    'random_label',         # Random Label
    'salun',                # SalUn
    'scrub',                # SCRUB
    'tarun',                # UNSIR
    'SVD',                  # SVD (gradient-free)
]

print(f'Methods ({len(RUN_METHODS)}): {RUN_METHODS}')
print(f'Seeds: {SPLIT_SEEDS}')
print(f'Total runs: {len(RUN_METHODS)} × {len(SPLIT_SEEDS)} = {len(RUN_METHODS)*len(SPLIT_SEEDS)}')

In [ ]:
def run_unlearn_regun(method, seed, split, start_ckpt):
    """
    Run one unlearning experiment on the 3:7 cross-class split.
    Returns dict with all evaluation metrics.
    """
    retain_indices = split['retain_indices']
    forget_indices = split['forget_indices']
    n_forget = len(forget_indices)
    n_retain = len(retain_indices)

    lr     = get_lr(method)
    epochs = get_epochs(method)

    kw = dict(
        unlearn_method=method,
        epochs_or_steps=epochs,
        lr=lr,
        batch_size=UNLEARN_BS,
        num_retain_samples=n_retain,
        num_forget_samples=n_forget,
        unlearn_class=[],   # cross-class split: no single class to forget
        remove_FC=False, CMFClassifier=True,
        grad_norm_clip=1.0,
    )
    if method in ('salun',):
        kw['salun_threshold'] = 0.5
    if method == 'SVD':
        s = _SVD[DATASET]
        kw.update(SVD_alpha_r=s['alpha_r'], SVD_alpha_f=s['alpha_f'],
                  SVD_samples=s['samples'], SVD_max_patches=s['max_patches'])
    if method == 'tarun':
        kw.update(tarun_impair_lr=1e-4 if DATASET=='cifar10' else 2e-4,
                  tarun_samples_per_class=1000)
    if method == 'scrub':
        kw.update(scrub_del_bsz=64, scrub_sgda_bsz=64,
                  scrub_msteps=2, scrub_epochs=epochs)

    args = make_args(**kw)

    # Build loaders from index sets
    retain_ds = SubSet(dataset_train, retain_indices)
    forget_ds = SubSet(dataset_train, forget_indices)
    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
    forget_loader = torch.utils.data.DataLoader(forget_ds, **LOADER_KW)

    m = get_model(args, device)
    m.load_state_dict(torch.load(start_ckpt, map_location=device))
    optimizer = optim.SGD(m.parameters(), lr=lr,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)

    print(f'\n{"-"*60}')
    print(f'  METHOD : {method}  SEED : {seed}')
    print(f'  forget_indices={n_forget}  retain_indices={n_retain}')
    print(f'  lr={lr}  epochs={epochs}')
    print(f'{"-"*60}')

    t0 = time.time()
    try:
        unlearnt = unlear_func[method](
            args=args, model=m, device=device,
            retain_loader=retain_loader,
            forget_loader=forget_loader,
            train_loader=train_loader,
            val_loader=None,
            test_loader=test_loader,
            optimizer=optimizer,
            epochs=epochs,
            train_dataset=dataset_train,
            val_index=np.arange(len(dataset_train)),
            test_forget_loader=forget_loader,
        )
    except Exception as e:
        print(f'  ERROR: {e}')
        return None

    elapsed = time.time() - t0

    # Save checkpoint
    ckpt_dir = f'{CKPT_ROOT_NB3}/{method}'
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_out = f'{ckpt_dir}/{DATASET}_{ARCH}_{_MODE_TAG}_seed{seed}.pt'
    torch.save(unlearnt.state_dict(), ckpt_out)

    # ── Output accuracy over INDEX SETS ───────────────────────────────
    unlearnt.eval()
    # Retain accuracy: eval on test set samples corresponding to retain classes
    # Since forget set spans all classes, we evaluate on all test samples
    # and report separately on forget/retain TRAIN indices as reference
    ra, _, _ = eval_on_indices(unlearnt, dataset_train, retain_indices, device)
    fa, _, _ = eval_on_indices(unlearnt, dataset_train, forget_indices, device)
    # Also get standard test accuracy
    test_ra, _, _ = eval_on_indices(unlearnt, dataset_test,
                                    list(range(len(dataset_test))), device)

    print(f'  → output: retain={ra:.4f}  forget={fa:.4f}  '
          f'test_acc={test_ra:.4f}  ({elapsed/60:.1f} min)')

    result = dict(
        method=method,
        seed=seed,
        output_retain_acc=ra,
        output_forget_acc=fa,
        test_acc=test_ra,
        wall_clock_minutes=elapsed/60,
        ckpt=ckpt_out,
    )

    # ── Probe evaluation ──────────────────────────────────────────────
    if not TEST_MODE:
        probe_res = run_probe_eval(
            unlearnt, retain_indices, forget_indices, retain_indices,
            dataset_train, dataset_test, device, NUM_CLASSES, args
        )
        result.update(probe_res)
        print(f'  → probe: retain={probe_res["probe_retain_acc"]:.4f}  '
              f'forget={probe_res["probe_forget_acc"]:.4f}')

    return result


print('run_unlearn_regun helper defined.')

In [ ]:
ALL_RESULTS = []

for seed in SPLIT_SEEDS:
    split = splits[seed]
    print(f'\n{"#"*65}')
    print(f'  SEED {seed}')
    print(f'{"#"*65}')

    for method in RUN_METHODS:
        res = run_unlearn_regun(method, seed, split, CKPT_PRETRAIN)
        if res is not None:
            ALL_RESULTS.append(res)

print(f'\nAll methods done. {len(ALL_RESULTS)} runs completed.')

## D. Evaluation Summary & Results CSV

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)

# Numeric columns for aggregation
_num_cols = [c for c in [
    'output_retain_acc', 'output_forget_acc', 'test_acc',
    'probe_retain_acc', 'probe_forget_acc', 'wall_clock_minutes'
] if c in results_df.columns]

summary = results_df.groupby('method')[_num_cols].agg(['mean', 'std']).round(4)
print(f'\n=== Group A Results — {DATASET}/{ARCH} (mean ± std over {len(SPLIT_SEEDS)} seeds) ===')
print(summary.to_string())

# ── Name map for display ──────────────────────────────────────────────
NAME_MAP = {
    'grad_ascent_descent': 'NegGrad+',
    'random_label': 'Random Label',
    'salun': 'SalUn',
    'scrub': 'SCRUB',
    'tarun': 'UNSIR',
    'SVD': 'SVD',
}
results_df['method_name'] = results_df['method'].map(NAME_MAP).fillna(results_df['method'])

# Save
csv_path = f'/kaggle/working/results_grpA_{DATASET}_{ARCH}.csv'
results_df.to_csv(csv_path, index=False)
print(f'\nResults saved: {csv_path}')

# ── Bar chart ─────────────────────────────────────────────────────────
if 'output_retain_acc' in results_df.columns:
    summary_plot = results_df.groupby('method')[['output_retain_acc','output_forget_acc']].mean()
    summary_plot.index = [NAME_MAP.get(i, i) for i in summary_plot.index]

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(summary_plot))
    w = 0.38
    ax.bar(x - w/2, summary_plot['output_retain_acc'], w,
           label='Retain Acc', color='steelblue', alpha=0.85)
    ax.bar(x + w/2, summary_plot['output_forget_acc'], w,
           label='Forget Acc', color='tomato', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(summary_plot.index, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.08)
    ax.set_title(f'Group A — 3:7 Cross-Class Split\n'
                 f'{DATASET}/{ARCH} (mean over {len(SPLIT_SEEDS)} seeds)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('/kaggle/working/chart_grpA.png', dpi=120)
    plt.show()
    print('Chart saved.')